In [12]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv
from groq import Groq
import langextract as lx

In [13]:
# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
MODEL_ID = "llama-3.1-8b-instant" 
csv_path = r"C:\Users\Customer\OneDrive - University of Bristol\War as Discourse\Truth Corpus Scrape\combined_test_corpus.csv"
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

In [32]:
# ==============================================================================
# 2. PROMPT
# ==============================================================================
system_prompt = """
You are a specialized extraction agent. Extract passages related to Iran.
Only extract if the post refers to Iran, Iranian actors, Tehran, the Islamic Republic, Ayatollahs, or the IRGC.

IMPORTANT: Return a JSON object with a key "extractions" containing a list of objects.
Each object must have:
"class": "about_Iran",
"text": "the exact text from the post"

CRITICAL RULES:
1. Extract the text EXACTLY as it appears.
2. IMPORTANT: If the extracted text contains double quotes ("), REPLACE them with single quotes (') to avoid JSON errors.
3. Do NOT wrap the extracted text in additional quotation marks.
4. If no match is found, return: {"extractions": []}

Example: {"extractions": [{"class": "about_Iran", "text": "He said, 'Iran is a great place,' and then left."}]}
"""

In [33]:
# 3. DATA PREPARATION & SORTING
# ==============================================================================
print("Loading and preparing CSV...")
df = pd.read_csv(csv_path)

# --- FIX 1: CHRONOLOGICAL SORTING ---
# Convert created_at to actual datetime objects so we can sort them correctly
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df = df.sort_values(by='created_at', ascending=True) # Oldest to Newest

# --- FIX 2: ROBUST COLUMN MAPPING ---
# Some CSVs use 'url', some use 'URL'. We'll normalize them all to lowercase.
df.columns = [c.lower() for c in df.columns]

# Define the mapping from our needed keys to the possible CSV column names
mapping = {
    "content": "content",
    "date": "created_at",
    "url": "url",
    "media": "media",
    "replies": "replies_count",
    "reblogs": "reblogs_count",
    "favs": "favourites_count"
}

# Fill missing values and ensure types are correct
for target, source in mapping.items():
    if source in df.columns:
        # For counts, we convert to float then int to remove the ".0" from floats
        if target in ["replies", "reblogs", "favs"]:
            df[source] = pd.to_numeric(df[source], errors='coerce').fillna(0).astype(int).astype(str)
        else:
            df[source] = df[source].fillna("N/A").astype(str)
    else:
        print(f"Warning: Column {source} not found in CSV. Setting to N/A.")
        df[source] = "N/A"

all_results = []
print("Done, move to processing documents via Groq...")

Loading and preparing CSV...
Done, move to processing documents via Groq...


In [34]:
# ==============================================================================
# 4. PROCESSING LOOP
# ==============================================================================
for i, row in df.iterrows():
    text = row[mapping["content"]]
    
    print(f"Analyzing Document {i}...", end=" ", flush=True)
    
    success = False
    retries = 3
    while not success and retries > 0:
        try:
            completion = client.chat.completions.create(
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": text}],
                model=MODEL_ID,
                response_format={"type": "json_object"},
                timeout=30.0
            )
            
            data = json.loads(completion.choices[0].message.content)
            items = data.get("extractions", []) if isinstance(data, dict) else []

            lx_extractions = []
            for item in items:
                ext_text = item.get("text", "").strip()
                if not ext_text: continue
                start_pos = text.lower().find(ext_text.lower())
                if start_pos != -1:
                    lx_extractions.append(lx.data.Extraction(
                        extraction_class=item.get("class", "about_Iran"),
                        extraction_text=ext_text,
                        char_interval={"start_pos": start_pos, "end_pos": start_pos + len(ext_text)}
                    ))
            
            doc = lx.data.Document(text=text, document_id=i)
            doc.extractions = lx_extractions 
            doc.metadata = {
                "date": row[mapping["date"]],
                "url": row[mapping["url"]],
                "media": row[mapping["media"]],
                "replies": row[mapping["replies"]],
                "reblogs": row[mapping["reblogs"]],
                "favs": row[mapping["favs"]]
            }
            all_results.append(doc)
            success = True 
            print("Done.")
            time.sleep(1.0) 

        except Exception as e:
            if "429" in str(e):
                print("Rate limit... sleeping...", end=" ")
                time.sleep(3)
                retries -= 1
            else:
                print(f"Error: {e}")
                doc = lx.data.Document(text=text, document_id=i)
                doc.extractions = []
                doc.metadata = {
                    "date": row[mapping["date"]], "url": row[mapping["url"]], "media": row[mapping["media"]],
                    "replies": row[mapping["replies"]], "reblogs": row[mapping["reblogs"]], "favs": row[mapping["favs"]]
                }
                all_results.append(doc)
                success = True

Analyzing Document 89... Done.
Analyzing Document 25... Done.
Analyzing Document 84... Done.
Analyzing Document 49... Done.
Analyzing Document 11... Done.
Analyzing Document 50... Done.
Analyzing Document 6... Done.
Analyzing Document 44... Done.
Analyzing Document 19... Done.
Analyzing Document 1... Done.
Analyzing Document 30... Done.
Analyzing Document 27... Done.
Analyzing Document 39... Done.
Analyzing Document 23... Done.
Analyzing Document 24... Done.
Analyzing Document 60... Done.
Analyzing Document 48... Done.
Analyzing Document 33... Done.
Analyzing Document 21... Done.
Analyzing Document 41... Done.
Analyzing Document 77... Done.
Analyzing Document 72... Done.
Analyzing Document 55... Done.
Analyzing Document 99... Done.
Analyzing Document 98... Done.
Analyzing Document 14... Done.
Analyzing Document 13... Done.
Analyzing Document 17... Done.
Analyzing Document 10... Done.
Analyzing Document 54... Done.
Analyzing Document 16... Done.
Analyzing Document 7... Done.
Analyzing D

In [35]:
# ==============================================================================
# 5. CUSTOM VISUALIZER
# ==============================================================================
def create_custom_viz(results, output_filename):
    viz_data = []
    for doc in results:
        # Format date for display (YYYY-MM-DD)
        date_str = str(doc.metadata["date"])
        clean_date = date_str.split('T')[0] if 'T' in date_str else date_str
        
        viz_data.append({
            "id": doc.document_id,
            "metadata": {**doc.metadata, "date": clean_date},
            "text": doc.text,
            "extractions": [
                {"class": ex.extraction_class, "text": ex.extraction_text, 
                 "start": ex.char_interval["start_pos"], "end": ex.char_interval["end_pos"]}
                for ex in doc.extractions
            ]
        })

    html_template = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Iran Analysis - Detailed Visualization</title>
        <style>
            body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: #f0f2f5; padding: 40px; color: #333; }}
            .container {{ max-width: 1100px; margin: auto; background: white; padding: 30px; border-radius: 15px; box-shadow: 0 4px 20px rgba(0,0,0,0.1); }}
            .header {{ text-align: center; margin-bottom: 20px; }}
            .meta-panel {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; background: #f8f9fa; padding: 15px; border-radius: 10px; border: 1px solid #ddd; margin-bottom: 20px; font-size: 14px; }}
            .meta-item {{ display: flex; flex-direction: column; }}
            .meta-label {{ font-weight: bold; color: #666; font-size: 12px; text-transform: uppercase; }}
            .meta-value {{ color: #1a73e8; text-decoration: none; word-break: break-all; }}
            .text-window {{ font-size: 18px; line-height: 1.6; padding: 20px; border: 1px solid #ddd; border-radius: 10px; white-space: pre-wrap; margin-bottom: 20px; min-height: 200px; background: #fff; }}
            .highlight {{ background-color: #fff59d; border-bottom: 3px solid #fbc02d; cursor: pointer; font-weight: bold; position: relative; }}
            .highlight:hover::after {{ content: attr(data-label); position: absolute; bottom: 125%; left: 50%; transform: translateX(-50%); background: #333; color: white; padding: 4px 8px; border-radius: 4px; font-size: 12px; z-index: 10; }}
            .controls {{ display: flex; justify-content: center; align-items: center; gap: 20px; margin-top: 20px; }}
            button {{ padding: 10px 20px; font-size: 16px; cursor: pointer; background: #1a73e8; color: white; border: none; border-radius: 5px; }}
            button:disabled {{ background: #ccc; }}
            .status {{ text-align: center; font-size: 14px; color: #888; margin-top: 10px; }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header"><h2>Iran Attributions Analysis</h2></div>
            <div class="meta-panel" id="metaPanel">
                <div class="meta-item"><span class="meta-label">Date</span><span id="mDate" class="meta-value">---</span></div>
                <div class="meta-item"><span class="meta-label">URL</span><a id="mUrl" class="meta-value" target="_blank">---</a></div>
                <div class="meta-item"><span class="meta-label">Media</span><a id="mMedia" class="meta-value" target="_blank">---</a></div>
                <div class,="meta-item"><span class="meta-label">Replies</span><span id="mReplies" class="meta-value">---</span></div>
                <div class="meta-item"><span class="meta-label">Reblogs</span><span id="mReblogs" class="meta-value">---</span></div>
                <div class="meta-item"><span class="meta-label">Favourites</span><span id="mFavs" class="meta-value">---</span></div>
            </div>
            <div id="textWindow" class="text-window">Loading...</div>
            <div class="controls">
                <button id="prevBtn" onclick="changeDoc(-1)">Previous</button>
                <button id="nextBtn" onclick="changeDoc(1)">Next</button>
            </div>
            <div id="status" class="status">Document 0 of 0</div>
        </div>
        <script>
            const data = {json.dumps(viz_data)};
            let currentIndex = 0;
            function renderDoc() {{
                if (data.length === 0) return;
                const doc = data[currentIndex];
                document.getElementById('mDate').innerText = doc.metadata.date;
                document.getElementById('mUrl').innerText = doc.metadata.url;
                document.getElementById('mUrl').href = doc.metadata.url.startsWith('http') ? doc.metadata.url : '#';
                document.getElementById('mMedia').innerText = doc.metadata.media;
                document.getElementById('mMedia').href = doc.metadata.media.startsWith('http') ? doc.metadata.media : '#';
                document.getElementById('mReplies').innerText = doc.metadata.replies;
                document.getElementById('mReblogs').innerText = doc.metadata.reblogs;
                document.getElementById('mFavs').innerText = doc.metadata.favs;
                let text = doc.text;
                const sortedEx = [...doc.extractions].sort((a, b) => b.start - a.start);
                let highlightedText = text;
                sortedEx.forEach(ex => {{
                    const before = highlightedText.substring(0, ex.start);
                    const target = highlightedText.substring(ex.start, ex.end);
                    const after = highlightedText.substring(ex.end);
                    highlightedText = `${{before}}<span class="highlight" data-label="${{ex.class}}">${{target}}</span>${{after}}`;
                }});
                document.getElementById('textWindow').innerHTML = highlightedText || "(Empty Document)";
                document.getElementById('status').innerText = `Document ${{currentIndex + 1}} of ${{data.length}}`;
                document.getElementById('prevBtn').disabled = (currentIndex === 0);
                document.getElementById('nextBtn').disabled = (currentIndex === data.length - 1);
            }}
            function changeDoc(dir) {{
                currentIndex += dir;
                renderDoc();
            }}
            renderDoc();
        </script>
    </body>
    </html>
    """
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write(html_template)

# ==============================================================================
# 5. EXECUTION
# ==============================================================================
create_custom_viz(all_results, "trump_truth_visualization.html")
print("SUCCESS: Visualization saved with chronological order and full metadata.")

SUCCESS: Visualization saved with chronological order and full metadata.
